In [1]:
################ Read the Landsat image using a shapefile ####
from datetime import datetime, timedelta
import os
import glob
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import rasterio
from rasterio.mask import mask

In [2]:
def extract_pixel_values(image_path, points, combined_mask=None):
    with rasterio.open(image_path) as src:
        # Get image bounds and shape
        bounds = src.bounds
        width = src.width
        height = src.height
        print('width and height')
        print(width)
        print(height)
        # Initialize a list to store the extracted values
        extracted_values = []
        
        for _, point in points.iterrows():
            # Convert point to pixel coordinates (row, col)
            x, y = point.geometry.x, point.geometry.y
            row, col = src.index(x, y)
            
            # Check if the point is within the bounds of the image
            if not (0 <= col < width and 0 <= row < height):
                #extracted_values.append(None)  # If out of bounds, append None
                continue
    
            # Read the pixel value for the first band (or change to the desired band)
            pixel_value = src.read(1)[row, col]
            # Apply combined mask if provided
            if combined_mask is not None:
                # Use the combined mask to check if the pixel is cloud or cirrus
                if combined_mask[row, col]:  # If masked (cloud or cirrus), exclude pixel
                    print('huma huma')
                else:
                    extracted_values.append([x, y, pixel_value])
            else:
                extracted_values.append([x, y, pixel_value])
            
    return extracted_values

In [3]:
def extract_pixel_values_L5(image_path, points, mask_path):
    with rasterio.open(image_path) as src:
        # Get image bounds and shape
        bounds = src.bounds
        width = src.width
        height = src.height
        print('width and height')
        print(width)
        print(height)
        # Initialize a list to store the extracted values
        extracted_values = []
        with rasterio.open(mask_path) as mask_src:
                mask_data = mask_src.read(1)  # Read the mask band
                mask_transform = mask_src.transform  # Transform for mask file
                mask_width = mask_src.width
                mask_height = mask_src.height
            
        for _, point in points.iterrows():
            # Convert point to pixel coordinates (row, col)
            x, y = point.geometry.x, point.geometry.y
            row, col = src.index(x, y)
            
            # Check if the point is within the bounds of the image
            if not (0 <= col < width and 0 <= row < height):
                #extracted_values.append(None)  # If out of bounds, append None
                continue
    
            # Read the pixel value for the first band (or change to the desired band)
            pixel_value = src.read(1)[row, col]
            # Apply combined mask if provided
            
            mask_row, mask_col = rasterio.transform.rowcol(mask_transform, x, y)
                
            # Check if the point is within the bounds of the mask file
            if not (0 <= mask_col < mask_width and 0 <= mask_row < mask_height):
                continue
            
            # Check the mask value
            if mask_data[mask_row, mask_col]:  # If masked (cloud or cirrus), exclude pixel
                a = 1
            else:
                extracted_values.append([x, y, pixel_value])
            
    return extracted_values

In [4]:
############# For loop - Providing a landsat image and the regional polygon to get all the points ######
sensor = 5
shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion.shp'  # Replace with the path to your shapefile
gdf = gpd.read_file(shapefile_path)
print(f'Shapefile CRS: {gdf.crs}')

input_folder = '/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222'  # Adjust this to your folder path
output_folder = '/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped'  # Adjust this to your output folder path

if sensor == 8:
    file_suffixes = ['QA_PIXEL.TIF', 'B2.TIF', 'B3.TIF', 'B4.TIF', 'B5.TIF', 'B6.TIF', 'B7.TIF']
else:
    file_suffixes = ['QA_PIXEL.TIF', 'B1.TIF', 'B2.TIF', 'B3.TIF', 'B4.TIF', 'B5.TIF', 'B7.TIF'] ## for LS 5
# Create output directory if it does not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Loop through each suffix and process matching Landsat images
for suffix in file_suffixes:
    print('the suffix is:')
    print(suffix)
    # Use glob to get all TIF files that match the suffix in the folder
    landsat_files = glob.glob(os.path.join(input_folder, f'*{suffix}'))
    
    # Process each file
    for landsat_path in landsat_files:
        with rasterio.open(landsat_path) as src:
            print(f'Processing {landsat_path}')
            print(f'Landsat CRS: {src.crs}')
            
            # Reproject the shapefile to match the Landsat image CRS (if necessary)
            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)
            
            # Step 3: Mask (Crop) the Raster with the Polygon
            # Convert the geometry to GeoJSON format (required by rasterio)
            polygon_geom = [feature["geometry"] for feature in gdf.__geo_interface__["features"]]
            
            # Use rasterio.mask.mask to crop the image
            cropped_image, cropped_transform = mask(src, polygon_geom, crop=True)
            
            # Copy metadata from the original image
            cropped_meta = src.meta.copy()
            cropped_meta.update({
                "driver": "GTiff",
                "height": cropped_image.shape[1],
                "width": cropped_image.shape[2],
                "transform": cropped_transform
            })
        
        # Step 4: Save the Cropped Image
        output_path = os.path.join(output_folder, os.path.basename(landsat_path))
        with rasterio.open(output_path, "w", **cropped_meta) as dest:
            dest.write(cropped_image)
        
        print(f'Cropped image saved at: {output_path}')


Shapefile CRS: epsg:4326
the suffix is:
QA_PIXEL.TIF
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/LT05_L2SP_015042_19860222_20200917_02_T1_QA_PIXEL.TIF
Landsat CRS: EPSG:32617
Cropped image saved at: /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_QA_PIXEL.TIF
the suffix is:
B1.TIF
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B1.TIF
Landsat CRS: EPSG:32617
Cropped image saved at: /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B1.TIF
the suffix is:
B2.TIF
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B2.TIF
Landsat CRS: EPSG:32617
Cropped image saved at: /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B2.TIF
the suffix is:
B3.TIF
Processing /Users/bz

In [5]:
###### This is for Landsat 5 #########################

qa_band = '/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_QA_PIXEL.TIF'

with rasterio.open(qa_band) as src:
    qa_band = src.read(1)  # Read the QA_PIXEL band
    
    transform = src.transform  # Get the GeoTransform
    crs = src.crs  # Get the Coordinate Reference System
    profile = src.profile  # Get the file profile
    # Mask for Cloud (Bit 1)
    cloud_mask = qa_band & 0x02  # Bitwise AND to extract cloud bits
    cloud_pixels = cloud_mask > 0  # Cloud pixels will be True (1), others False (0)
    
    # Mask for Cirrus (Bit 4)
    cirrus_mask = qa_band & 0x10  # Bitwise AND to extract cirrus cloud bits
    cirrus_pixels = cirrus_mask > 0  # Cirrus pixels will be True (1), others False (0)
    
    # Mask for Shadow (Bit 3)
    shadow_mask = qa_band & 0x08  # Bitwise AND to extract shadow bits
    shadow_pixels = shadow_mask > 0  # Shadow pixels will be True (1), others False (0)

    # Combine the masks to exclude cloud, cirrus, and shadow
    combined_mask = cloud_pixels | cirrus_pixels | shadow_pixels # True for unwanted pixels

    # Optionally, invert the mask to get valid pixels
    valid_pixels = ~combined_mask  # True for valid pixels

    # Count pixels
    print(f"Number of cloud/cirrus/shadow pixels: {np.sum(combined_mask)}")
    print(f"Number of valid pixels: {np.sum(valid_pixels)}")

    profile.update(
        dtype=rasterio.uint8,  # Data type for binary mask
        count=1,  # Single band
        compress='lzw'  # Optional: Compression
    )

    # Write the binary mask to a GeoTIFF
    with rasterio.open('mask.tif', 'w', **profile) as dst:
        dst.write(combined_mask.astype(rasterio.uint8), 1)  # Write combined_mask as uint8

Number of cloud/cirrus/shadow pixels: 43094
Number of valid pixels: 847708


In [6]:
### collecting the pixel values based on the shapefile and cropped images #####
## save as a list ###
point_shapefile_path = 'intersected_pixel_points.shp'  # Replace with your pixel center points shapefile path
gdf_points = gpd.read_file(point_shapefile_path)

all_extracted_values = []  

if sensor == 8:
    suffixes = ['B2.TIF', 'B3.TIF', 'B4.TIF', 'B5.TIF', 'B6.TIF', 'B7.TIF']
else:
    suffixes = ['B1.TIF', 'B2.TIF', 'B3.TIF', 'B4.TIF', 'B5.TIF', 'B7.TIF'] ## just for LS5
    
mask_path = '/Users/bz294/Documents/WhiteZone/Code/mask.tif'

for suffix in suffixes:
    landsat_files = glob.glob(os.path.join(output_folder, f'*{suffix}'))
    print('the file now is:')
    print(landsat_files)
    for landsat_path in landsat_files:
        print(f'Processing {landsat_path}')

        extracted_values = extract_pixel_values_L5(landsat_path, gdf_points, mask_path)
        
        # Save the extracted results (example: to CSV)
        extracted_data = np.array(extracted_values)
        all_extracted_values.append(extracted_data)





the file now is:
['/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B1.TIF']
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B1.TIF
width and height
1227
726
the file now is:
['/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B2.TIF']
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B2.TIF
width and height
1227
726
the file now is:
['/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B3.TIF']
Processing /Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T1_SR_B3.TIF
width and height
1227
726
the file now is:
['/Users/bz294/Documents/WhiteZone/LS/Spatial/1985/19860222/cropped/LT05_L2SP_015042_19860222_20200917_02_T

In [7]:
for i, val in enumerate(all_extracted_values):
    print(f"Element at index {i}: Type={type(val)}")
    if isinstance(val, (np.ndarray, list)):  # Check if the element is a list or an ndarray
        print(f"  Shape/Length: {getattr(val, 'shape', len(val))}")  # Get shape if ndarray, length if list
    else:
        print("  Not a list or ndarray, it's a scalar.")

Element at index 0: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)
Element at index 1: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)
Element at index 2: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)
Element at index 3: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)
Element at index 4: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)
Element at index 5: Type=<class 'numpy.ndarray'>
  Shape/Length: (172049, 3)


In [8]:
####### creating three indices ###########
all_extracted_values = np.array(all_extracted_values)

b6 = np.array(all_extracted_values[4])  # Convert to NumPy array if it's not already

# Extract only the third index of the second axis (rows dimension), i.e., all rows, all columns, and index 2
b6_slice = b6[:, 2]

b5 = np.array(all_extracted_values[3])  # Convert to NumPy array if it's not already

# Extract only the third index of the second axis (rows dimension), i.e., all rows, all columns, and index 2
b5_slice = b5[:, 2]

b4 = np.array(all_extracted_values[2])  # Convert to NumPy array if it's not already

# Extract only the third index of the second axis (rows dimension), i.e., all rows, all columns, and index 2
b4_slice = b4[:, 2]


# Perform elementwise operation
ndvi = (b5_slice - b4_slice) / (b5_slice + b4_slice)
ndmi = (b5_slice - b6_slice) / (b5_slice + b6_slice)
savi = 1.5 * (b5_slice - b4_slice) / (b5_slice + b4_slice + 0.5)

features = all_extracted_values[:, :, 2]

features = features.T
ndvi = ndvi.reshape(-1, 1)
ndmi = ndmi.reshape(-1, 1)
savi = savi.reshape(-1, 1)

feat_all = np.concatenate((features, ndvi, ndmi, savi), axis=1)

np.save('feat_all.npy', feat_all)

coord = all_extracted_values[0, :, :2]
np.save('coord_all.npy', coord)

/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_66425/1667914863.py:21: RuntimeWarning: invalid value encountered in divide
  ndvi = (b5_slice - b4_slice) / (b5_slice + b4_slice)
/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_66425/1667914863.py:22: RuntimeWarning: invalid value encountered in divide
  ndmi = (b5_slice - b6_slice) / (b5_slice + b6_slice)


In [ ]:
############# Providing a landsat image and the regional polygon to get all the points ######

shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion.shp'  # Replace with the path to your shapefile
gdf = gpd.read_file(shapefile_path)
print(f'Shapefile CRS: {gdf.crs}')

landsat_path = '/Users/bz294/Documents/WhiteZone/LS/Spatial/20171212/LC08_L2SP_015042_20130402_20200912_02_T1_SR_B1.TIF'  # Replace with the path to your Landsat image

with rasterio.open(landsat_path) as src:
    print(f'Landsat CRS: {src.crs}')
    
    # Reproject the shapefile to match the Landsat image CRS (if necessary)
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)
    
    # Step 3: Mask (Crop) the Raster with the Polygon
    # Convert the geometry to GeoJSON format (required by rasterio)
    polygon_geom = [feature["geometry"] for feature in gdf.__geo_interface__["features"]]
    
    # Use rasterio.mask.mask to crop the image
    cropped_image, cropped_transform = mask(src, polygon_geom, crop=True)
    
    # Copy metadata from the original image
    cropped_meta = src.meta.copy()
    cropped_meta.update({
        "driver": "GTiff",
        "height": cropped_image.shape[1],
        "width": cropped_image.shape[2],
        "transform": cropped_transform
    })

# Step 4: Save the Cropped Image
output_path = 'cropped_landsat_image.tif'  # Define your output path
with rasterio.open(output_path, "w", **cropped_meta) as dest:
    dest.write(cropped_image)

print(f'Cropped image saved at: {output_path}')


In [ ]:
####### get the points from the cropped image ####
### but the points covers a square, need to further subsetted. #####

nrows, ncols = cropped_image.shape[1], cropped_image.shape[2]

# Get the x, y coordinates of the pixel centers
xs, ys = np.meshgrid(
    np.arange(0, ncols),  # Column indices (for x-direction)
    np.arange(0, nrows)   # Row indices (for y-direction)
)

# # Calculate the center of each pixel (half-pixel shift)
# xs = xs + 0.5
# ys = ys + 0.5

# Transform pixel indices (row, col) to spatial coordinates (x, y)
# Use the cropped image's transform to calculate real-world coordinates
pixel_x_coords, pixel_y_coords = rasterio.transform.xy(cropped_transform, ys, xs)

# Flatten the x, y coordinates and pixel values
pixel_x_coords = np.array(pixel_x_coords).flatten()
pixel_y_coords = np.array(pixel_y_coords).flatten()
pixel_values = cropped_image[0].flatten()  # Assuming single-band image (use [0] for first band)

# Step 6: Create Geopandas GeoDataFrame for the Pixel Center Points
# Filter out NoData values (assuming NoData is masked as np.nan or specific value)
valid_pixels = ~np.isnan(pixel_values)  # Adjust this if NoData value is not NaN
points = [
    Point(x, y) for x, y, val in zip(pixel_x_coords[valid_pixels], 
                                     pixel_y_coords[valid_pixels], 
                                     pixel_values[valid_pixels])
]

# Create GeoDataFrame with pixel coordinates and pixel values
gdf_points = gpd.GeoDataFrame({
    'pixel_value': pixel_values[valid_pixels],
    'geometry': points
}, crs=src.crs)

# Step 7: Save the Pixel Center Points as a MultiPoint Shapefile
output_shapefile_path = 'pixel_center_points.shp'
gdf_points.to_file(output_shapefile_path, driver='ESRI Shapefile')

print(f'Pixel center points shapefile saved at: {output_shapefile_path}')

In [ ]:
### interesect the points with regional polygon to fit for the shape of the region ####

point_shapefile_path = 'pixel_center_points.shp'  # Replace with your pixel center points shapefile path
gdf_points = gpd.read_file(point_shapefile_path)

shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion.shp'  # Replace with the path to your shapefile
gdf_polygon = gpd.read_file(shapefile_path)

# Step 3: Ensure both shapefiles have the same CRS
if gdf_points.crs != gdf_polygon.crs:
    gdf_polygon = gdf_polygon.to_crs(gdf_points.crs)

# Step 4: Perform the Intersection of the Point and Polygon Shapefiles
gdf_intersection = gpd.overlay(gdf_points, gdf_polygon, how='intersection')

# Step 5: Save the Resulting Intersected Points as a New Shapefile
output_intersected_shapefile_path = 'intersected_pixel_points.shp'
gdf_intersection.to_file(output_intersected_shapefile_path, driver='ESRI Shapefile')

print(f'Intersected points saved at: {output_intersected_shapefile_path}')

In [ ]:
coord = all_extracted_values[:, 0, :2]
#np.save('feat_all.npy', feat_all)

In [ ]:
all_extracted_values.shape

In [ ]:
np.save('coord_all.npy', coord)

In [ ]:
features = all_extracted_values[:, :, 2]
features.shape

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(ndvi.flatten(), bins=50, color='green', edgecolor='black', alpha=0.7)

# Add labels and title
plt.title('Histogram of NDVI', fontsize=14)
plt.xlabel('NDVI Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# Show the plot
plt.grid(True)
plt.show()

In [ ]:
output_folder = '/Users/bz294/Documents/WhiteZone/LS/Spatial/20130402/cropped/'  # Adjust this to your output folder path
landsat_files = output_folder + 'LC08_L2SP_015042_20130402_20200912_02_T1_SR_B4.TIF'

point_shapefile_path = 'intersected_pixel_points.shp'  # Replace with your pixel center points shapefile path
gdf_points = gpd.read_file(point_shapefile_path)
print(f'Processing {landsat_files}')

b4 = extract_pixel_values(landsat_files, gdf_points, combined_mask)

# Save the extracted results (example: to CSV)
#extracted_data = np.array(list(zip(extracted_values)))
#all_extracted_values.append(extracted_data)

#print(f"Extracted values saved to {output_file}")

In [ ]:
array_3_slice[0]

In [ ]:
all_extracted_values[3]

In [ ]:
b4

In [ ]:
b5

In [ ]:
shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/2013/pt_filter.shp'  # Replace with the path to your shapefile
gdf = gpd.read_file(shapefile_path)
print(f'Shapefile CRS: {gdf.crs}')
print(f'Number of Features in Shapefile: {len(gdf)}')

In [ ]:
shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/2013_revised_dec16.shp'  # Replace with the path to your shapefile
gdf = gpd.read_file(shapefile_path)
print(f'Shapefile CRS: {gdf.crs}')
print(f'Number of Features in Shapefile: {len(gdf)}')